# 01 - Deploy the Foundry infrastructure

Before we explore ASSERT, we need models it can talk to. This notebook prepares a Foundry project and three models to compare. You only need to do this once for your lab environment.

Complete [setup](./00_setup.ipynb) and select **Python (assert01-foundry-evaluation)** first. Deployment creates Azure resources; model calls later in the lab incur usage charges. Global Standard can process requests outside your account's region. See the [run guide](../README.md) for costs and cleanup.

## Load the demo tools

Run this cell to load the helpers and prepare a folder for the lab's saved files. Keep the notebook in its `notebooks` directory.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from azure.identity import AzureCliCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from azure.mgmt.resource.resources import ResourceManagementClient
import pandas as pd
from IPython.display import FileLink, display

demo_root = Path('..').resolve()
sys.path.insert(0, str(demo_root))
from app.assert_lab import DeploymentMetadata, ModelDeployment, artifact_path, validate_catalog, write_connections
from app.notebook_helpers import resolve_az_cli

az_cmd = resolve_az_cli()
if az_cmd is None:
    raise FileNotFoundError('Azure CLI is required; run the setup notebook first.')
generated = artifact_path(root=demo_root)
generated.mkdir(parents=True, exist_ok=True)
print(f'Generated artifacts: {generated}')

## Confirm the subscription and three model selections

Choose where the lab will run: set `AZURE_SUBSCRIPTION_ID` or enter your subscription ID below, using the same subscription you selected during setup. Check the resource-group name and region too.

The labels `older_model`, `intermediate_model`, and `newer_model` identify the three models we will compare. Keep the supplied selections for your first walkthrough; the next step checks whether Azure can deploy them.

In [ ]:
SUBSCRIPTION_ID = os.environ.get('AZURE_SUBSCRIPTION_ID', '')
RESOURCE_GROUP = 'rg-assert01'
LOCATION = 'eastus2'
PROVIDER_API_VERSION = '2024-10-21'
MODEL_PLAN = (
    ModelDeployment('older_model', 'assert-older', 'gpt-5', '2025-08-07', 'GlobalStandard', 10),
    ModelDeployment('intermediate_model', 'assert-intermediate', 'gpt-5.4', '2026-03-05', 'GlobalStandard', 10),
    ModelDeployment('newer_model', 'assert-newer', 'gpt-5.6-sol', '2026-07-09', 'GlobalStandard', 10),
)
SKIPPED_MODEL = None
SKIP_REASON = None
MODEL_SELECTIONS = tuple(model for model in MODEL_PLAN if model != SKIPPED_MODEL)
if not SUBSCRIPTION_ID.strip():
    raise ValueError('Set SUBSCRIPTION_ID explicitly or set AZURE_SUBSCRIPTION_ID before provisioning.')
credential = AzureCliCredential()
cognitive = CognitiveServicesManagementClient(credential, SUBSCRIPTION_ID)
resources = ResourceManagementClient(credential, SUBSCRIPTION_ID)
print(f'Selected subscription: {SUBSCRIPTION_ID}; group: {RESOURCE_GROUP}; region: {LOCATION}')
if SKIPPED_MODEL is not None:
    print(f'Explicitly skipped {SKIPPED_MODEL.alias}: {SKIP_REASON}')

## Check the live model catalog and quota

Check that the selected models are available and your subscription has enough quota. If this stops with a quota error, free unused allocation or request more quota before continuing.

In [ ]:
catalog = list(cognitive.models.list(LOCATION))
usages = list(cognitive.usages.list(LOCATION))
model_checks = validate_catalog(MODEL_SELECTIONS, catalog, usages, skipped=SKIPPED_MODEL)
display(pd.DataFrame(model_checks))

## Review inherited policy before provisioning

Your organization may restrict which Azure resources you can create. This is **Azure Policy**, not the assistant behavior policy we will test with ASSERT.

Follow the [Azure Policy review guide](../../../.github/skills/azure-demo-authoring/references/workflow.md#scoped-azure-policy-review) for your subscription and resource group, with your administrator if needed. Confirm that a public endpoint, keyless sign-in, and the selected region/models are allowed. After your review, set `POLICY_REVIEW_COMPLETE = True` in its own cell below and run it. Then run the following cell to save the prerequisite record.

In [ ]:
POLICY_REVIEW_COMPLETE = True

In [ ]:
if not POLICY_REVIEW_COMPLETE:
    raise ValueError('Complete the scoped inherited-policy review above before provisioning; then acknowledge it here.')
preflight_path = artifact_path('preflight.json', root=demo_root)
preflight_path.write_text(json.dumps({
    'checked_at': datetime.now(timezone.utc).isoformat(),
    'subscription_id': SUBSCRIPTION_ID, 'resource_group': RESOURCE_GROUP,
    'location': LOCATION, 'policy_review_complete': True, 'models': model_checks,
}, indent=2) + '\n', encoding='utf-8')
print(f'Non-secret prerequisite record: {preflight_path}')
display(FileLink((Path('..') / preflight_path.relative_to(demo_root)).as_posix()))

## Prepare the dedicated resource group

Create the resource group that will hold this lab's Azure resources. Keep it dedicated to the lab so cleanup is straightforward.

In [ ]:
resource_group = resources.resource_groups.create_or_update(RESOURCE_GROUP, {'location': LOCATION})
print(f'Resource group ready: {resource_group.name}')

## Identify the signed-in developer for inference access

Identify your signed-in Azure user so the deployment can give you permission to call the models without API keys. Use the same subscription as setup; your account needs permission to create role assignments.

In [ ]:
context = subprocess.run([az_cmd, 'account', 'show', '--query', 'id', '--output', 'tsv'], capture_output=True, text=True)
if context.returncode != 0:
    raise RuntimeError(f'Azure context lookup failed\nstderr: {context.stderr.strip()}\nstdout: {context.stdout.strip()}')
if context.stdout.strip() != SUBSCRIPTION_ID:
    raise ValueError('Select the confirmed subscription in setup before deploying or using native Entra authentication.')
principal = subprocess.run([az_cmd, 'ad', 'signed-in-user', 'show', '--query', 'id', '--output', 'tsv'], capture_output=True, text=True)
if principal.returncode != 0:
    raise RuntimeError(f'Signed-in-user lookup failed\nstderr: {principal.stderr.strip()}\nstdout: {principal.stdout.strip()}')
INFERENCE_PRINCIPAL_ID = principal.stdout.strip()
if not INFERENCE_PRINCIPAL_ID:
    raise ValueError('The signed-in developer lookup returned no object ID.')
print('The signed-in developer will receive account-scoped inference access through Bicep.')

## Build and deploy the copied Bicep

The first cell prepares the deployment template. The next creates the Foundry account, project, model deployments, and your model-access permission. Deployment can take a few minutes; wait for it to finish before continuing.

In [ ]:
template_path = demo_root / 'infra' / 'main.bicep'
parameters_path = artifact_path('deployment.parameters.json', root=demo_root)
parameters_path.write_text(json.dumps({
    '$schema': 'https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#',
    'contentVersion': '1.0.0.0',
    'parameters': {
        'location': {'value': LOCATION},
        'inferencePrincipalId': {'value': INFERENCE_PRINCIPAL_ID},
        'modelDeployments': {'value': [model.to_dict() for model in MODEL_SELECTIONS]},
    },
}, indent=2) + '\n', encoding='utf-8')
build_result = subprocess.run(
    [az_cmd, 'bicep', 'build', '--file', str(template_path), '--outfile', str(artifact_path('main.json', root=demo_root))],
    capture_output=True, text=True, cwd=demo_root,
)
if build_result.returncode != 0:
    raise RuntimeError(f'az bicep build failed (exit {build_result.returncode})\nstderr: {build_result.stderr.strip()}\nstdout: {build_result.stdout.strip()}')
if build_result.stderr.strip():
    print(build_result.stderr.strip())
print('Bicep build succeeded; this does not prove deployment or provider compatibility.')

In [ ]:
deployment_result = subprocess.run(
    [az_cmd, 'deployment', 'group', 'create', '--subscription', SUBSCRIPTION_ID,
     '--resource-group', RESOURCE_GROUP, '--name', 'assert01-infrastructure',
     '--template-file', str(template_path), '--parameters', '@' + str(parameters_path), '--output', 'json'],
    capture_output=True, text=True, cwd=demo_root,
)
if deployment_result.returncode != 0:
    raise RuntimeError(f'az deployment group create failed (exit {deployment_result.returncode})\nstderr: {deployment_result.stderr.strip()}\nstdout: {deployment_result.stdout.strip()}')
deployment_outputs = json.loads(deployment_result.stdout)['properties']['outputs']
print('Infrastructure deployment succeeded. Export actual metadata next; no account keys were ARM outputs.')

## Generate non-secret connection settings

Save the deployed model addresses and names for the remaining notebooks. Authentication uses your Azure sign-in, so there are no API keys to copy into the lab.

In [ ]:
account_name = deployment_outputs['aiServicesAccountName']['value']
account = cognitive.accounts.get(RESOURCE_GROUP, account_name)
if account.properties.disable_local_auth is not True or account.properties.public_network_access != 'Enabled':
    raise ValueError('Deployed authentication/network settings differ from Bicep; resolve upstream policy/template drift before connecting.')
actual_models = []
for selected in MODEL_SELECTIONS:
    actual = cognitive.deployments.get(RESOURCE_GROUP, account_name, selected.deployment_name)
    observed = ModelDeployment(selected.alias, actual.name, actual.properties.model.name, actual.properties.model.version, actual.sku.name, actual.sku.capacity)
    if observed != selected or actual.properties.version_upgrade_option != 'NoAutoUpgrade':
        raise ValueError(f'Actual deployment differs from the explicit selection: {selected.alias}')
    actual_models.append(observed)
metadata = DeploymentMetadata(
    subscription_id=SUBSCRIPTION_ID, resource_group=RESOURCE_GROUP, location=account.location,
    account_name=account.name, account_id=account.id, ai_endpoint=account.properties.endpoint,
    project_name=deployment_outputs['foundryProjectName']['value'],
    project_endpoint=deployment_outputs['foundryProjectEndpoint']['value'],
    api_base=deployment_outputs['azureOpenAIEndpoint']['value'], api_version=PROVIDER_API_VERSION,
    deployments=tuple(actual_models), skipped_deployment=SKIPPED_MODEL, skip_reason=SKIP_REASON,
)
env_path, metadata_path = write_connections(metadata, root=demo_root)
print(f'Generated ignored non-secret environment: {env_path}')
print(f'Generated non-secret metadata: {metadata_path}')
for path in (env_path, metadata_path):
    display(FileLink((Path('..') / path.relative_to(demo_root)).as_posix()))

## Continue with Connect models

The resources are deployed. Open [02_configure.ipynb](./02_configure.ipynb) for the final setup step: checking that ASSERT can connect to the models. You will choose the behavior policy in the ASSERT flow notebook after setup is complete.